### Understanding Structured Output, Tools, and Agents Through Event Booking Example

The relationship between **Structured Output**, **Tools**, and **Agents** can be understood as an incremental evolution of an LLM application.

<img src="../../assets/eventbot_mermaid.png" width="800" height="200">


### Structured Output → Tools → Agents

This flow represents the evolution of an LLM application from simple response to an Agent.

- **Structured Output** enables the LLM to convert user input into a predictable schema.  
  *Example: EventBot extracts event_name, Customer name, and ticket counts from a booking request.*

- **Tools** extend the LLM's capabilities by allowing it to interact with custom functions.  
  *Example: EventBot uses tools to see different Schemas and decide to respond with appropriate schema based on user query*

- **Agents** combine LLM reasoning, Structured Output, and Tools to create an system that can take actions.
  *Example: EventBot understands the booking request, checks availability, and completes the reservation.*


In [36]:

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain_core.tools import tool
from typing import Union


load_dotenv()

True

In [7]:
gpt_5_nano = ChatOpenAI(model="gpt-5-nano")
gpt_3_5_turbo = ChatOpenAI(model="gpt-3.5-turbo")

print(f"Initialize gpt_5_nano with structured output support: {gpt_5_nano.profile['structured_output']}")
print(f"Initialize gpt_3_5_turbo with structured output support: {gpt_3_5_turbo.profile['structured_output']}")

Initialize gpt_5_nano with structured output support: True
Initialize gpt_3_5_turbo with structured output support: False


#### Tool Strategy & Provider Strategy

Two different mechanisms achieve the same guarantee.
* `ProviderStrategy` uses the model provider's own native structured-output feature (fast, but only works where supported).
* `ToolStrategy` fakes it via a synthetic tool call (works almost everywhere, slightly slower).

#### Provider Strategy vs Tool Strategy
LangChain supports two strategies for generating **Structured Outputs**. 

| Strategy | How it Works | When to Use |
|----------|--------------|-------------|
| **ProviderStrategy** | Uses the model provider's native Structured Output capability to generate and validate responses against the schema. | Preferred when the selected model supports native Structured Outputs (e.g., supported OpenAI models). |
| **ToolStrategy** | Uses the model's tool calling capability by creating an internal (synthetic) tool from provided schema. The tool is **not executed**—its arguments are extracted as the structured response. | Used when the model does not support native Structured Outputs but supports tool calling. |

**Key Differences**

- **ProviderStrategy**
  - More reliable because the provider enforces the schema.
  - Lower chance of malformed or invalid responses.
  - Available only for models with native Structured Output support.

- **ToolStrategy**
  - Works with most modern tool-calling models.
  - Uses the model's tool-calling interface internally to produce structured data.
  - Slightly less reliable than ProviderStrategy, as schema validation is handled by LangChain rather than the model provider.

### Walkthrough these concepts using BookingEvent as example

In [37]:

class BookingRequest(BaseModel):
    customer_name: str = Field(description="The customer's name")
    event_name: str = Field(description="The event they want to go")
    action: Literal["book", "cancel"] = Field(description="Whether this is a new booking or a cancellation")
    ticket_count: int = Field(description="How many tickets, default 1 if not mentioned", default=1)

#### 1. Call to `gpt-3.5-turbo` with ProviderStrategy will fail
```python
BadRequestError: Error code: 400 - {'error': {'message': "Invalid parameter: 'response_format' of type 'json_schema' is not supported with this model.
```

In [ ]:
agent = create_agent(model="openai:gpt-3.5-turbo", response_format=ProviderStrategy(BookingRequest)) 
result = agent.invoke({"messages": [{"role": "user", "content": "My name is Shivam, I booked for Fifa worldcup ticket which i want to cancel"}]})
print(result["structured_response"]) # Will fail in agent.invoke()


### Why did this error occur?

1. **`gpt-3.5-turbo` does not support native Structured Outputs.**  
*ProviderStrategy* relies on the model provider's native **response_format=json_schema** feature. Since **gpt-3.5-turbo** doesn't support this, OpenAI API rejects the request with a **400 BadRequestError**.
2. **`ProviderStrategy` was explicitly forced.**  
By specifying **response_format=ProviderStrategy(BookingRequest)**, LangChain does not automatically fall back to `ToolStrategy`.

#### 2. Call to `gpt-3.5-turbo` with ToolStrategy will be Successfull
**How the Agent executes it**
When we invoke this agent, the langchain handles the structured data pipeline automatically:
* The agent instructs gpt-3.5-turbo that it must call a specific tool (e.g., named BookingRequest) matching our schema.
* gpt-3.5-turbo triggers a standard tool call.
* The create_agent architecture intercepts this tool call, parses the arguments against our Pydantic class, and saves the verified object under the structured_response key.

In [ ]:
agent = create_agent(model="openai:gpt-3.5-turbo", response_format=ToolStrategy(BookingRequest))
result = agent.invoke({"messages": [{"role": "user", "content": "My name is Shivam, I booked for Fifa worldcup ticket which i want to cancel"}]})
print(f"Customer Name: {result['structured_response'].customer_name}")
print(f"Event Name: {result['structured_response'].event_name}")
print(f"Action: {result['structured_response'].action}")    
print(f"Ticket Count: {result['structured_response'].ticket_count}")

Customer Name: Shivam
Event Name: Fifa worldcup
Action: cancel
Ticket Count: 1


### Invoke `gpt_3_5_turbo` model using `with_structured_output`.
* When we map BookingRequest schema it by defaults uses ProviderStrategy.
* We expect this call would fail as gpt_3_5_turbo model doesn't support structured_output.

In [42]:

structured_llm = gpt_3_5_turbo.with_structured_output(BookingRequest)
result = structured_llm.invoke('Is Interstellar showing tonight? Book 2 seats for Shivam')
print(f"Customer Name: {result.customer_name}")
print(f"Event Name: {result.event_name}")
print(f"Action: {result.action}")    
print(f"Ticket Count: {result.ticket_count}")

/Users/geetikabatra/genai_bootcamp/implementation/langchain-learning/.venv/lib/python3.13/site-packages/langchain_openai/chat_models/base.py:2471: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


Customer Name: Shivam
Event Name: Interstellar
Action: book
Ticket Count: 2


* Above request works fine because with_structured_output binds the model to the BookingRequest schema.
* Model is able to generate structured output provided as function arguments for tool call.
* structured_llm = llm.with_structured_output(BookingRequest) is conceptually equivalent to gpt_3_5_turbo.bind_tools([BookingRequest])

```python
         Prompt
            │
            ▼
      LangChain
            │
            ├── Convert Pydantic model → Tool schema  (.bind_tools([BookingRequest]))
            │
            ├── Send tool definition to OpenAI
            │
            ▼
      GPT-3.5 Turbo
            │
            └── Returns tool call: { customer_name: "Shivam", event_name: "Interstellar", ticket_count: 2, action: "book"}
            │
            ▼
      LangChain parses tool arguments
            │
            ▼
      BookingRequest(...)
```

### The Problem: What If a Message Could Be More Than One Kind of Thing?
* `BookingRequest` assumes we already know it's a booking. 
* Real support systems see many kinds of requests — what if a customer might have ten different intents: book, cancel, modify, shift, check, and more? 
* Forcing everything through one schema stops making sense.

### The Fix: `Union` — Letting the Model Choose Which Schema Fits

**Analogy:**: Two different intake forms at a service counter. 
* A "New Booking" form.
* A "Cancellation" form. 

The customer picks whichever form actually matches their situation.

**Union types:** 
* Multiple schema options for **Tool Strategy**. 
* The model will choose the most appropriate schema based on the context.

#### Defining 2 Schema for 2 diverent kind of user queries.
1. To book a new ticket of an Event.
2. To cancel teh existing booked event.

In [ ]:
class NewBooking(BaseModel):
    """A request to book NEW tickets."""
    customer_name: str
    event_name: str
    ticket_count: int = Field(description="Number of tickets, must be between 1 and 10", ge=1, le=10, default=1)

class CancelBooking(BaseModel):
    """A request to CANCEL an existing booking."""
    customer_name: str
    event_name: str


#### Lets try to Cancel the Event and see if it responds correct **CancelBooking** Schema

In [ ]:
union_agent = create_agent(
    model='openai:gpt-5-nano',
    tools=[],
    response_format=ToolStrategy(Union[NewBooking, CancelBooking])
)

result = union_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "My name is Shivam, I booked for Fifa worldcup ticket which i want to cancel."
        }
    ]
})
print(f"Customer Name: {result['structured_response'].customer_name}")
print(f"Event Name: {result['structured_response'].event_name}")

Customer Name: Shivam
Event Name: Fifa worldcup ticket


#### Lets try to Book new the Event and see if it responds correct **NewBooking** Schema
**Issue without System Prompt:** If we don't specify instruction, it hallucinates and give **customer_name** we didn't mentioned in our query

In [ ]:
book_result = union_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Book 5 tickets for comedy show"
        }
    ]
})
print(f"Customer Name: {book_result['structured_response'].customer_name}") 
print(f"Event Name: {book_result['structured_response'].event_name}")
print(f"Ticket Count: {book_result['structured_response'].ticket_count}")

Customer Name: Guest
Event Name: Comedy Show
Ticket Count: 5


**Added System Prompt**: Specifying to dont make up any information, so **customer_name** should be empty/None

In [ ]:
union_agent_system_prompt = create_agent(
    model='openai:gpt-5-nano',
    tools=[],
    response_format=ToolStrategy(Union[NewBooking, CancelBooking]),
    system_prompt="""You are a helpful booking assistant, dont make up any information, just return the structured output as per the schema only if the user provides it.
                    If the user does not provide a customer name, return None or empty for that field."""
)

book_result = union_agent_system_prompt.invoke({
    "messages": [{
            "role": "user",
            "content": "Book 5 tickets for comedy show"}
        ]
    })
print(f"Customer Name: {book_result['structured_response'].customer_name}") # No customer name provided in the request, so this will be None or empty.
print(f"Event Name: {book_result['structured_response'].event_name}")
print(f"Ticket Count: {book_result['structured_response'].ticket_count}")

Customer Name: 
Event Name: comedy show
Ticket Count: 5


**Conditional Flow:** 
Check at runtime what kind of request we got, and take action based on that.

In [ ]:
if isinstance(result["structured_response"], NewBooking):
    print("We got a new booking! Ticket count:", result["structured_response"].ticket_count)
elif isinstance(result["structured_response"], CancelBooking):
    print("We got a cancellation request.")

### Error handling
- Models can make mistakes when generating structured output via tool calling. 
- LangChain provides intelligent retry mechanisms to handle these errors automatically.
- ToolStrategy(NewBooking, handle_errors=...)
    1. handle_errors: bool = True (default)
    2. handle_errors: bool = False - Returns error is any validation gets failed.
    3. handle_errors: str = 'Custom error message' -  Tries to fix issue by propagating custom Exception message to LLM.

**In our example we will validate to book number of tickets between 1-10**:

We have declared this in NewBooking:  
```python 
ticket_count: int = Field(description="Number of tickets, must be between 1 and 10", ge=1, le=10, default=1)
```

**Lets try to book 30 new tickets, that should would fail our validations**

#### handle_errors=True
- Iteratively tries fix issues by propagating Exception message to LLM.

> LLM did unexpected thing.
> It provided to make 3 NewBooking calls with 10 tickets each, making my request to book 30 tickets.😢 
> However Langchain denied that request and responded with below 
> ```python
>   Error: Model incorrectly returned multiple structured responses (NewBooking, NewBooking, NewBooking) when only one is expected.
```

In [ ]:
booking_agent_with_error_handling = create_agent(
    model='openai:gpt-5-nano',
    tools=[],
    response_format=ToolStrategy(NewBooking),
    system_prompt= "You are a helpful assistant that supports event bookings."
)

result = booking_agent_with_error_handling.invoke({
    "messages": [{"role": "user", "content": "I am Shivam, book 30 Fifa world cup tickets"}]})
result

{'messages': [HumanMessage(content='I am Shivam, book 30 Fifa world cup tickets', additional_kwargs={}, response_metadata={}, id='03ad24d2-c95f-4f49-9fb9-06f56df34b59'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1075, 'prompt_tokens': 192, 'total_tokens': 1267, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 960, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E5YlPjuO3vq4FhoLsrmwGvpidEFSG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f99ee-8af3-7241-9f5a-361b0e784747-0', tool_calls=[{'name': 'NewBooking', 'args': {'customer_name': 'Shivam', 'event_name': 'Fifa world cup', 'ticket_count': 10}, 'id': 'call_f10b3PMtu2lLaHNRkS

```text
Agent Execution Result
=====================

User Input
----------
"I am Shivam, book 30 Fifa world cup tickets"


Step 1: LLM Initial Response
----------------------------

Model:
gpt-5-nano-2025-08-07

Problem Detected
----------------

LangChain Validation Error:

Error:
"Model incorrectly returned multiple structured responses 
(NewBooking, NewBooking, NewBooking) when only one is expected."

Reason:
The model returned 3 structured responses instead of a single
NewBooking object.

The request was:
"Book 30 tickets"

The model incorrectly split the request into:
- Booking 10 tickets
- Booking 10 tickets
- Booking 10 tickets


Step 2: LangChain Error Feedback to Model
-----------------------------------------

LangChain sent the error back to the model:

"Please fix your mistakes."


Step 3: Model Retry Response
----------------------------

The model generated a single structured output:

Tool Call:
----------

Tool: NewBooking

Arguments:
{
    "customer_name": "Shivam",
    "event_name": "Fifa world cup",
    "ticket_count": 10
}

Step 4: Structured Response Generated
-------------------------------------

Tool Result:

NewBooking(
    customer_name="Shivam",
    event_name="Fifa world cup",
    ticket_count=10
)

Final Agent Output
------------------

structured_response:
{
    "customer_name": "Shivam",
    "event_name": "Fifa world cup",
    "ticket_count": 10
}
```

#### handle_errors=False
- Langchain doesn't try to fix issues and responds with validation error.

In [ ]:
booking_agent_without_error_handling = create_agent(
    model='openai:gpt-5-nano',
    tools=[],
    response_format=ToolStrategy(NewBooking, handle_errors=False),
    system_prompt= "Extract the booking details exactly as stated, Don't invent anything"
)

result = booking_agent_without_error_handling.invoke({
    "messages": [{"role": "user", "content": "I am Shivam, book 30 Fifa world cup tickets"}]})

```python
  ValidationError: 1 validation error for NewBooking
  ticket_count
  ...
  StructuredOutputValidationError: Failed to parse structured output for tool 'NewBooking': Failed to parse data to NewBooking: 1 validation error for NewBooking
  ticket_count
    Input should be less than or equal to 10 [type=less_than_equal, input_value=30, input_type=int]
```

In [ ]:
booking_agent_with_custom_err_msg = create_agent(
    model='openai:gpt-5.5-nano',
    tools=[],
    response_format=ToolStrategy(NewBooking, handle_errors="Ticket count should not be greater than 10"),
    system_prompt= "You are a helpful assistant that supports event bookings."
)

result = booking_agent_without_error_handling.invoke({
    "messages": [{"role": "user", "content": "I am Shivam, book 30 Fifa world cup tickets moake sure do not book less that 30 tickets"}]})